In [3]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '../src')

from preprocessing import (
    concatenate_columns,
    select_columns_remove_unselected,
    print_value_counts,
    prepare_labels,
    save_dataframe,
    rename_columns,
    stratified_kfold_cross_validation
)

In [ ]:
datasets_config = [
    {
        'input_file': './dataset/B-02-dataset-balanceado.csv',
        'dataset_config': [
            {
                'description': '60B/40M & No duplicates - Benign / 14 Types (All)',
                'labels': {
                    "Benign": "0",
                    "DDoS attacks-LOIC-HTTP": "1",
                    "DDOS attack-HOIC": "2",
                    "DoS attacks-Hulk": "3",
                    "Bot": "4",
                    "Infilteration": "5",
                    "SSH-Bruteforce": "6",
                    "DoS attacks-GoldenEye": "7",
                    "DoS attacks-Slowloris": "8",
                    "DDOS attack-LOIC-UDP": "9",
                    "Brute Force -Web": "10",
                    "Brute Force -XSS": "11",
                    "SQL Injection": "12",
                    "DoS attacks-SlowHTTPTest": "13",
                    "FTP-BruteForce": "14"
                },
                'output_file': './dataset/B-03-preparing-dataset-pt.csv'
            }
        ]
    }
]

for dataset_config in datasets_config:
    input_file = dataset_config['input_file']
    print('Input file: ' + input_file)
    for single_config in dataset_config['dataset_config']:
        description = single_config['description']
        labels = single_config['labels']
        output_file = single_config['output_file']
        print(description)
        df_built = pd.read_csv(input_file)
        df_prepared = (df_built.
                pipe(concatenate_columns, 'LABEL', 'source_text').
                pipe(select_columns_remove_unselected, ['source_text', 'LABEL']).
                pipe(print_value_counts, 'LABEL').
                pipe(rename_columns, {'LABEL': 'target_text'})
            )
        split_files = stratified_kfold_cross_validation(df_prepared, 
                                          5,
                                          0.2,
                                          list(labels.keys()), 
                                          output_file[:-4] + '-', 
                                          '.csv')
        print('Output file: ' + output_file)
        df_prepared = (df_prepared.
                pipe(select_columns_remove_unselected, ['source_text', 'target_text']).
                pipe(prepare_labels, 'target_text', labels).
                pipe(print_value_counts, 'target_text').
                pipe(save_dataframe, output_file)
            )
        for split_file in split_files:
            print('Split file: ' + split_file)
            df_split_prepared = pd.read_csv(split_file)
            df_split_prepared = (df_split_prepared.
                pipe(prepare_labels, 'target_text', labels).
                pipe(print_value_counts, 'target_text').
                pipe(save_dataframe, split_file)
            )
        print()
    print('\n\n')

Input file: ./dataset/A-02-dataset-balanceado.csv
60B/40M & No duplicates & Protocol - Benign / 14 Types (All)
